In [ ]:
import pandas as pd
import numpy as np
from ortools.linear_solver import pywraplp

# =========================
# PARAMETERS
# =========================

AVAILABLE_HOURS = 22
SETUP_HOURS = 40 / 60

SHORTAGE_PENALTY = 100
IDLE_PENALTY = 20
CHANGEOVER_PENALTY = 5
HOLDING_PENALTY = 2

# =========================
# LOAD FILES
# =========================

book_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_13march_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

hz_parts = pd.read_excel(book_path, sheet_name="HZ")
vt_parts = pd.read_excel(book_path, sheet_name="VT")

stats = pd.read_excel(book_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix")

# =========================
# MERGE DATA
# =========================

data = stats.merge(daily, left_on="Part", right_on="Material")

# =========================
# PRODUCTION RATE
# =========================

data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]

data = data[data["Rate"].notna()]

In [ ]:
# =========================
# PARTS
# =========================

parts = data["Material"].unique().tolist()

# =========================
# MACHINES
# =========================

machines = hz_matrix.columns[1:].tolist()

In [ ]:
# =========================
# INVENTORY
# =========================

inventory = dict(
    zip(data["Material"], data["Inventory on 24th"])
)

# =========================
# DEMAND
# =========================

demand = dict(
    zip(data["Material"], data["2026-03-12 Total Production Plan"])
)

# =========================
# RATE
# =========================

rate = dict(
    zip(data["Material"], data["Rate"])
)

In [ ]:
compatibility = {}

for _, row in hz_matrix.iterrows():

    part = row["Part"]

    for machine in machines:

        compatibility[(part, machine)] = row[machine]

In [ ]:
solver = pywraplp.Solver.CreateSolver("SCIP")

In [ ]:
x = {}

for p in parts:
    for m in machines:

        if compatibility.get((p,m),0) == 1:

            x[p,m] = solver.NumVar(0, solver.infinity(), f"x_{p}_{m}")

In [ ]:
shortage = {}

for p in parts:

    shortage[p] = solver.NumVar(0, solver.infinity(), f"short_{p}")

In [ ]:
for p in parts:

    production = solver.Sum(
        x[p,m] for m in machines if (p,m) in x
    )

    solver.Add(
        inventory[p] + production + shortage[p] >= demand[p]
    )

In [ ]:
for m in machines:

    runtime = solver.Sum(
        x[p,m] / rate[p]
        for p in parts
        if (p,m) in x
    )

    solver.Add(runtime <= AVAILABLE_HOURS)

In [ ]:
for m in machines:

    runtime = solver.Sum(
        x[p,m] / rate[p]
        for p in parts
        if (p,m) in x
    )

    solver.Add(runtime <= AVAILABLE_HOURS)

In [ ]:
status = solver.Solve()

print("Status:", status)

In [ ]:
plan = []

for (p,m), var in x.items():

    qty = var.solution_value()

    if qty > 0:

        plan.append({
            "Part": p,
            "Machine": m,
            "Qty": round(qty,0),
            "Run_Hours": round(qty / rate[p],2)
        })

plan_df = pd.DataFrame(plan)

plan_df.to_excel("Smart_APS_Plan_V1.xlsx", index=False)